In [3]:
# Step 1: Load & Inspect Data

import pandas as pd

# Load dataset
df = pd.read_excel("C:/Users/pooja/Documents/Job Applications/Data Analyst/Projects/Healthcare/ER-Wait-Time-Project/data/raw/ER_Wait_Time_Dataset.xlsx")

# Standardize column names
df.columns = (
    df.columns
      .str.strip()                                     # Remove leading/trailing spaces
      .str.lower()                                     # Convert to lowercase
      .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)  # Replace all non-alphanumeric with underscore
      .str.strip("_")                                  # REMOVE leading/trailing underscores
)

# Optional: check the new column names
print(df.columns)

# Quick check
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

Index(['visit_id', 'patient_id', 'hospital_id', 'hospital_name', 'region',
       'visit_date', 'day_of_week', 'season', 'time_of_day', 'urgency_level',
       'nurse_to_patient_ratio', 'specialist_availability',
       'facility_size_beds', 'time_to_registration_min', 'time_to_triage_min',
       'time_to_medical_professional_min', 'total_wait_time_min',
       'patient_outcome', 'patient_satisfaction'],
      dtype='str')
(5000, 19)
               visit_id patient_id hospital_id                 hospital_name  \
0  HOSP-1-20240210-0001  PAT-00001      HOSP-1  Springfield General Hospital   
1  HOSP-3-20241128-0001  PAT-00002      HOSP-3  Northside Community Hospital   
2  HOSP-3-20240930-0002  PAT-00003      HOSP-3  Northside Community Hospital   
3  HOSP-2-20240227-0001  PAT-00004      HOSP-2      Riverside Medical Center   
4  HOSP-1-20240306-0002  PAT-00005      HOSP-1  Springfield General Hospital   

  region          visit_date day_of_week  season   time_of_day urgency_level  \


In [4]:
# Step 2: Feature Engineering
# 1. Create Weekend Flag

df['weekend'] = df['day_of_week'].isin(['Saturday','Sunday']).astype(int)

In [6]:
# 2. Convert Time of Day to Ordered Categories

time_order = {'Morning':1, 'Afternoon':2, 'Evening':3, 'Night':4}
df['time_of_day_num'] = df['time_of_day'].map(time_order)

In [7]:
# 3. Break Down Wait Stages

df['reg_pct'] = df['time_to_registration_min'] / df['total_wait_time_min'] * 100
df['triage_pct'] = df['time_to_triage_min'] / df['total_wait_time_min'] * 100
df['medical_pct'] = df['time_to_medical_professional_min'] / df['total_wait_time_min'] * 100

In [8]:
# 4. Bucket Total Wait Times

wait_bins = [0, 15, 30, 60, df['total_wait_time_min'].max()]
wait_labels = ['0-15', '15-30', '30-60', '60+']
df['wait_time_bucket'] = pd.cut(df['total_wait_time_min'], bins=wait_bins, labels=wait_labels)

In [9]:
# 5. Standardize Urgency Levels

df['urgency_level'] = df['urgency_level'].str.title()

In [10]:
# Quick look at first 5 rows
print(df.head())

# Check data types
print(df.dtypes)

# Check for any remaining missing values
print(df.isnull().sum())

# Summary statistics for numeric columns
print(df.describe())

               visit_id patient_id hospital_id                 hospital_name  \
0  HOSP-1-20240210-0001  PAT-00001      HOSP-1  Springfield General Hospital   
1  HOSP-3-20241128-0001  PAT-00002      HOSP-3  Northside Community Hospital   
2  HOSP-3-20240930-0002  PAT-00003      HOSP-3  Northside Community Hospital   
3  HOSP-2-20240227-0001  PAT-00004      HOSP-2      Riverside Medical Center   
4  HOSP-1-20240306-0002  PAT-00005      HOSP-1  Springfield General Hospital   

  region          visit_date day_of_week  season   time_of_day urgency_level  \
0  Urban 2024-02-10 20:20:56    Saturday  Winter  Late Morning        Medium   
1  Rural 2024-11-28 02:07:47    Thursday    Fall       Evening        Medium   
2  Rural 2024-09-30 04:02:28      Monday    Fall       Evening           Low   
3  Urban 2024-02-27 00:31:13     Tuesday  Winter       Evening          High   
4  Urban 2024-03-06 16:52:26   Wednesday  Spring     Afternoon           Low   

   ...  time_to_medical_professional_m

In [19]:
# Define path for cleaned data
cleaned_path = "C:/Users/pooja/Documents/Job Applications/Data Analyst/Projects/Healthcare/ER-Wait-Time-Project/data/cleaned/df_cleaned.csv"

# Save to excel
df.to_csv(cleaned_path, index=False, encoding="utf-8")  # index=False avoids saving the row numbers

print(f"Cleaned data saved at: {cleaned_path}")

Cleaned data saved at: C:/Users/pooja/Documents/Job Applications/Data Analyst/Projects/Healthcare/ER-Wait-Time-Project/data/cleaned/df_cleaned.csv
